In [42]:
import pandas as pd
import numpy as np
import os
import mo_gymnasium as mo_gym
import gymnasium as gym
import sys
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
import arviz as az
import pymc as pm
import pyperclip

sys.path.append(os.path.abspath('..'))
from env.user_sim import UserSimEnv
import env.simulate as sim_utils
import utils
import process_data as dp
import mdp_utils
import pickle
import bayesian_analysis as ba


%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
evaluation_cols = ["time_spent", "likedness", "usefulness", "difficulty", "PAY_next", "expert_score", "diversity"]
evaluation_names = ["Time Spent", "Likedness", "Perceived Usefulness", "Difficulty", "Return Willingness", "Expert Usefulness", "Diversity"]

simulation_folder = "C:\\Users\\shirl\\Documents\\Studie\\2025-2026\\Thesis\\personalized-coping-challenges\\results\\pimorld_P=32_T=5_nO=5_linear_combined\\simulations"
simulation_files = [f for f in os.listdir(simulation_folder) if f.endswith('.pkl')]
all_sim_results = []
seen_random = False
for sim_file in simulation_files:
    with open(os.path.join(simulation_folder, sim_file), 'rb') as f:
        sim_result = pickle.load(f)
        if 'Multipolicy random' in sim_result['policy'].iloc[0]:
            if seen_random:
                sim_result['policy'] = sim_result['policy'].apply(lambda x: x.replace('Multipolicy random', 'Multipolicy random expert'))
            else:
                seen_random = True

        all_sim_results.append(sim_result)
combined_df = pd.concat(all_sim_results, ignore_index=True)

policy_abbreviations = {
    "Multipolicy random": "MORL-RC",
    "Multipolicy random expert": "MORL-RE",
    "Multipolicy expert_priority": "MORL-EP",
    "Random": "B-Rnd",
    "Equal Weights Policy": "B-Eq",
    "Correlation-based Policy": "B-Corr"
}

combined_df['policy'] = combined_df['policy'].map(policy_abbreviations)



In [25]:
sim_utils.interactive_plot_objective(all_sim_results, evaluation_names)

interactive(children=(Dropdown(description='Objective:', options=(('Time Spent', 0), ('Likedness', 1), ('Perce…

In [ ]:
reward_df = pd.DataFrame(
    combined_df["rewards"].tolist(),
    columns=evaluation_names
)

reward_df["policy"] = combined_df["policy"]
reward_df["user"] = combined_df["user"]

In [ ]:
user_driven_returns = ["Time Spent", "Likedness", "Perceived Usefulness", "Difficulty"]

user_means = reward_df.groupby(["policy", "user"])[user_driven_returns].mean()
policy_means, policy_stds = reward_df.groupby("policy")[user_driven_returns].mean(), reward_df.groupby("policy")[user_driven_returns].std()
improvement_from_random = policy_means.subtract(policy_means.loc["B-Rnd"], axis=1).div(policy_means.loc["B-Rnd"], axis=1) * 100

summary = policy_means.round(2).astype(str) + " ($\\uparrow$ " + improvement_from_random.round(2).astype(str) + "\\%)"

display(summary)
latex = summary.to_latex(escape=False, 
                       caption="Mean of user-driven returns for each policy across simulations, alongside the improvement compared to the random baseline policy.", 
                       label="tab:reward_summary_user_driven")
latex = latex.replace("\\begin{table}", "\\begin{table}[ht]\n\\centering")
pyperclip.copy(latex)

,Time Spent,Likedness,Perceived Usefulness,Difficulty
policy,,,,
B-Corr,8.73 ($\uparrow$ 96.17\%),6.24 ($\uparrow$ 93.01\%),4.44 ($\uparrow$ 90.31\%),3.24 ($\uparrow$ 57.09\%)
B-Eq,8.02 ($\uparrow$ 80.1\%),6.04 ($\uparrow$ 86.88\%),4.27 ($\uparrow$ 83.01\%),3.05 ($\uparrow$ 47.5\%)
B-Rnd,4.45 ($\uparrow$ 0.0\%),3.23 ($\uparrow$ 0.0\%),2.33 ($\uparrow$ 0.0\%),2.07 ($\uparrow$ 0.0\%)
MORL-EP,2.73 ($\uparrow$ -38.73\%),2.22 ($\uparrow$ -31.45\%),1.5 ($\uparrow$ -35.65\%),1.28 ($\uparrow$ -37.84\%)
MORL-RC,6.97 ($\uparrow$ 56.49\%),5.01 ($\uparrow$ 55.14\%),3.6 ($\uparrow$ 54.01\%),2.69 ($\uparrow$ 30.33\%)
MORL-RE,2.83 ($\uparrow$ -36.41\%),2.28 ($\uparrow$ -29.46\%),1.62 ($\uparrow$ -30.67\%),1.3 ($\uparrow$ -37.18\%)


In [64]:
comparison_policies = ["MORL-RC", "MORL-RE", "MORL-EP", "B-Eq", "B-Corr"]
for return_name in user_driven_returns[1:2]:
    user_means_baseline = user_means.loc["B-Rnd", return_name]
    user_means_comparison = user_means.loc[comparison_policies, return_name]
    for policy in comparison_policies:
        comparison = user_means_comparison.loc[policy]
        print(f"Comparison: {policy} vs B-Rnd for {return_name}")
        ba.two_sample_t_test(comparison, user_means_baseline)


Comparison: MORL-RC vs B-Rnd for Likedness
              mean     sd  hdi_2.5%  hdi_97.5%
improvement  1.784  0.046     1.696      1.875
effect_size  2.548  0.090     2.368      2.722
group1_mean  5.015  0.033     4.953      5.082
group2_mean  3.231  0.032     3.165      3.291
P(improvement > 0): 1.0000
P(Effect Size > 0.1): 1.0000
P(Effect Size > 0.2): 1.0000
P(Effect Size > 0.3): 1.0000
Comparison: MORL-RE vs B-Rnd for Likedness
              mean     sd  hdi_2.5%  hdi_97.5%
improvement -0.953  0.045    -1.037     -0.859
effect_size -1.365  0.073    -1.512     -1.226
group1_mean  2.278  0.031     2.219      2.342
group2_mean  3.231  0.031     3.170      3.293
P(improvement > 0): 0.0000
P(Effect Size > 0.1): 1.0000
P(Effect Size > 0.2): 1.0000
P(Effect Size > 0.3): 1.0000
Comparison: MORL-EP vs B-Rnd for Likedness
              mean     sd  hdi_2.5%  hdi_97.5%
improvement -1.024  0.044    -1.109     -0.935
effect_size -1.508  0.077    -1.657     -1.354
group1_mean  2.207  0.031     2.

In [ ]:
expert_driven_returns = ["Return Willingness", "Expert Usefulness", "Diversity"]
user_means = reward_df.groupby(["policy", "user"])[expert_driven_returns].mean()
policy_means, policy_stds = reward_df.groupby("policy")[expert_driven_returns].mean(), reward_df.groupby("policy")[expert_driven_returns].std()
improvement_from_random = policy_means.subtract(policy_means.loc["B-Rnd"], axis=1).div(policy_means.loc["B-Rnd"], axis=1) * 100

summary = policy_means.round(2).astype(str) + " ($\\uparrow$ " + improvement_from_random.round(2).astype(str) + "\\%)"

display(summary)
latex = summary.to_latex(escape=False, 
                       caption="Mean of expert-driven returns for each policy across simulations, alongside the improvement compared to the random baseline policy.", 
                       label="tab:reward_summary_expert_driven")
latex = latex.replace("\\begin{table}", "\\begin{table}[ht]\n\\centering")
pyperclip.copy(latex)

                Return Willingness           Expert Usefulness  \
policy                                                           
B-Corr     3.1 ($\uparrow$ 8.79\%)   0.63 ($\uparrow$ 68.47\%)   
B-Eq      3.04 ($\uparrow$ 6.86\%)    0.6 ($\uparrow$ 60.32\%)   
B-Rnd      2.85 ($\uparrow$ 0.0\%)     0.38 ($\uparrow$ 0.0\%)   
MORL-EP  2.75 ($\uparrow$ -3.59\%)  0.23 ($\uparrow$ -38.37\%)   
MORL-RC   3.01 ($\uparrow$ 5.74\%)   0.54 ($\uparrow$ 43.07\%)   
MORL-RE   3.03 ($\uparrow$ 6.27\%)  0.27 ($\uparrow$ -28.27\%)   

                          Diversity  
policy                               
B-Corr    0.0 ($\uparrow$ -100.0\%)  
B-Eq     0.18 ($\uparrow$ -53.64\%)  
B-Rnd       0.39 ($\uparrow$ 0.0\%)  
MORL-EP   0.73 ($\uparrow$ 85.91\%)  
MORL-RC   0.44 ($\uparrow$ 13.53\%)  
MORL-RE   0.69 ($\uparrow$ 77.65\%)  
\begin{table}
\caption{Mean of expert-driven returns for each policy across simulations, alongside the improvement compared to the random baseline policy.}
\label{tab:

In [51]:
T = 28
completion_per_user = (
    combined_df
    .sort_values(["policy", "user", "t"])
    .groupby(["policy", "user"])
    .tail(1)
)[["policy", "user", "num_completed"]]
completion_per_user["adherence"] = completion_per_user["num_completed"].div(T)

adherence_per_policy = completion_per_user.groupby("policy")["adherence"].mean()
improvement_from_random = adherence_per_policy.subtract(adherence_per_policy.loc["B-Rnd"]).div(adherence_per_policy.loc["B-Rnd"]) * 100
adherence_summary = adherence_per_policy.round(2).astype(str) + " ($\\uparrow$ " + improvement_from_random.round(2).astype(str) + "\\%)"
display(adherence_summary)
latex = adherence_summary.to_latex(escape=False, multicolumn=False,
                       caption="Mean adherence for each policy across simulations, alongside the improvement compared to the random baseline policy.", 
                       label="tab:adherence_summary")
latex = latex.replace("\\begin{table}", "\\begin{table}[ht]\n\\centering")
pyperclip.copy(latex)

policy
B-Corr      0.88 ($\uparrow$ 74.64\%)
B-Eq        0.86 ($\uparrow$ 70.91\%)
B-Rnd          0.5 ($\uparrow$ 0.0\%)
MORL-EP    0.33 ($\uparrow$ -35.41\%)
MORL-RC     0.76 ($\uparrow$ 50.04\%)
MORL-RE    0.38 ($\uparrow$ -24.65\%)
Name: adherence, dtype: str

In [60]:
# analyze per time state, the time spent on the activity
reward_df["time_state"] = combined_df["state"].apply(lambda x: x[1])

mean_time_spent_per_user = reward_df.groupby(["policy", "user", "time_state"])["Time Spent"].mean().reset_index()
mean_time_spent_per_policy_time = mean_time_spent_per_user.groupby(["policy", "time_state"])["Time Spent"].mean().unstack()
display(mean_time_spent_per_policy_time)

time_state,0,1
policy,,
B-Corr,7.536370,11.310110
B-Eq,7.277582,9.768301
B-Rnd,4.025605,5.418726
MORL-EP,2.133110,4.031409
MORL-RC,5.999427,9.084847
MORL-RE,2.253162,4.040346
